In [2]:
import pandas as pd
import sys
import os
import scanpy as sc

In [3]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [ ]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_44920\2864242217.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [20]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [29]:
os.chdir(r"G:\My Drive\result\publication\cellreport\revision\GLIPH2")
names = ['TCRab CD4','TCRab CD8aa','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

In [28]:
clone_counts

,subject:condition,clone_code,clone frequency,TRAV,TRBV,cdr3a,cdr3b,clone_id
0,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-1 CAVNTNAGKSTF CASSLPNEKLFF,1,TRAV1-1,TRBV11-1,CAVNTNAGKSTF,CASSLPNEKLFF,clonotype6
1,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAAHTNAGKSTF CASTSRDRGLHEQYF,1,TRAV1-1,TRBV11-2,CAAHTNAGKSTF,CASTSRDRGLHEQYF,clonotype7
2,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVETNAGKSTF CASSDRGQGGANVLTF,2,TRAV1-1,TRBV11-2,CAVETNAGKSTF,CASSDRGQGGANVLTF,clonotype9
3,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVSTNAGKSTF CASSVPNEKLFF,3,TRAV1-1,TRBV11-2,CAVSTNAGKSTF,CASSVPNEKLFF,clonotype11
4,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVYSGYSTLTF CASSLVIAGTSDTQYF,1,TRAV1-1,TRBV11-2,CAVYSGYSTLTF,CASSLVIAGTSDTQYF,clonotype12
...,...,...,...,...,...,...,...,...
6707,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV28 CAVATGANNLFF CAAARGYGNQPQHF,1,TRAV8-6,TRBV28,CAVATGANNLFF,CAAARGYGNQPQHF,clonotype17694
6708,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVALKTSYDKVIF CASSSRVNRETQYF,1,TRAV8-6,TRBV5-1,CAVALKTSYDKVIF,CASSSRVNRETQYF,clonotype17753
6709,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVRISGGSYIPTF CASSFLYNSPLHF,1,TRAV8-6,TRBV5-1,CAVRISGGSYIPTF,CASSFLYNSPLHF,clonotype17763
6710,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV6-6 CAVSDLYNAGNMLTF CASKGTSDTEAFF,1,TRAV8-6,TRBV6-6,CAVSDLYNAGNMLTF,CASKGTSDTEAFF,clonotype17843


In [26]:
adata_ab.obs['general type']

AAACCTGAGCGCCTCA-1-3-PB       TCRab CD4
AAACCTGAGGCAAAGA-1-3-PB     TCRab CD8ab
AAACCTGAGTTATCGC-1-3-PB       TCRab CD4
AAACCTGCAAGCGATG-1-3-PB       TCRab CD4
AAACCTGGTTACGTCA-1-3-PB       TCRab CD4
                               ...     
TTTGTCAAGAACTGTA-1-5-IEL    TCRab CD8ab
TTTGTCAAGGGAGTAA-1-5-IEL    TCRab CD8ab
TTTGTCAAGTTGTCGT-1-5-IEL      TCRab CD4
TTTGTCACAATGACCT-1-5-IEL    TCRab CD8ab
TTTGTCACATCGATTG-1-5-IEL    TCRab CD8ab
Name: general type, Length: 26297, dtype: category
Categories (3, object): ['TCRab CD4', 'TCRab CD8aa', 'TCRab CD8ab']